# EEExport for HLSL30 and HLSS30

|Band name|OLI Band number|MSI Band number|HLS Band code name|Wavelength (micrometers)|
|:---:|:---:|:---:|:---:|:---:|
|Coastal Aerosol|1|1|CA|0.43-0.45*|
|Blue|2|2|BLUE|0.45-0.51*|
|Green|3|3|GREEN|0.53-0.59*|
|Red|4|4|RED|0.64-0.67*|
|Red-Edge 1|-|5|RE1|0.69-0.71**|
|Red-Edge 2|-|6|RE2|0.73-0.75**|
|Red-Edge 3|-|7|RE3|0.77-0.79**|
|NIR Narrow|5|8A|NIR1|0.85-0.88*|
|NIR Broad|-|8|NIR2|0.78-0.88**|
|SWIR 1|6|11|SWIR1|1.57-1.65*|
|SWIR 2|7|12|SWIR2|2.11-2.29*|
|Water Vapor|-|9|WV|0.93-0.95**|
|Cirrus|9|10|CIRRUS|1.36-1.38*|
|Thermal Infrared 1|10|-|TIRS1|10.60-11.19*|

> $*$ from OLI specifications (may vary for S10 product which follow MSI specifications)
>
> $**$ from MSI specifications

In [ ]:
retry_list = [
    "NASA_HLS_v002_GREEN_lon94.2605_lat29.7733_part3",
]

# Define coordinates as a list of [lon, lat]
coordinates_list = [
    [94.2604595275288, 29.773270070797224],  # 尼西村
    [100.112018, 25.653985], # 苍山
    [104.259534, 31.217034], # 德阳
    [91.911327, 29.181582], # 山南
    [92.603540, 28.944296], # 加查
    [94.361009, 29.648327], # 林芝
    [91.273398, 29.790422], # 拉萨
    [104.659162, 29.258789], # 自贡
    [115.897694, 33.007419], # 阜阳
    [116.337168, 31.356277], # 南岳
    [118.165609, 30.137470], # 黄山
    [102.269126, 27.962762], # 凉山
    [101.266320, 24.368124], # 景东
    [100.717270, 22.129849], # 西双版纳
    [109.041607, 24.310168], # 柳州
    # Add more coordinates here
]

# Image Size in pixels
pixel_size = 1000

image_collection_hlsl = "NASA/HLS/HLSL30/v002"
image_collection_hlss = "NASA/HLS/HLSS30/v002"

map_hlsl = {
    "BLUE": "B2",
    "GREEN": "B3",
    "RED": "B4",
    "NIR1": "B5",
    "SWIR1": "B6",
    "SWIR2": "B7",
    "QA": "Fmask"
}

map_hlss = {
    "BLUE": "B2",
    "GREEN": "B3",
    "RED": "B4",
    "NIR1": "B8A",
    "SWIR1": "B11",
    "SWIR2": "B12",
    "QA": "Fmask"
}

bands_to_export_hlsl = list(map_hlsl.values())
bands_to_export_hlss = list(map_hlss.values())

export_qa = False # Set to True to include the QA band in the export
export_bands = ["BLUE", "GREEN", "RED", "NIR1", "SWIR1", "SWIR2"]
if export_qa:
    export_bands.append("QA")

start_date = "2015-01-01"
end_date = "2026-12-31"
export_folder = "EEExport"
export_scale = 30 # HLS use 30m resolution
max_bands_per_export = 4500 # Ensure we stay under 5000


In [ ]:
import ee
from ee.feature import Feature
from ee.featurecollection import FeatureCollection
from ee.geometry import Geometry
from ee.imagecollection import ImageCollection
from ee.image import Image
from ee.batch import Export
from ee.filter import Filter
from ee.join import Join

try:
    ee.Initialize(project="ee-yangluhao990714")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="ee-yangluhao990714")

In [ ]:
def mask_hls_clouds(image):
    qa = image.select("QA")
    # Bit 0 is cirrus, Bit 1 is cloud, Bit 2 is adjacent to cloud/shadow, Bit 3 is cloud shadow
    cirrus_mask = qa.bitwiseAnd(1 << 0).eq(0)
    cloud_mask = qa.bitwiseAnd(1 << 1).eq(0)
    adjacent_cloud_mask = qa.bitwiseAnd(1 << 2).eq(0)
    shadow_mask = qa.bitwiseAnd(1 << 3).eq(0)
    return image.updateMask(cirrus_mask.And(cloud_mask).And(shadow_mask).And(adjacent_cloud_mask))

for coords in coordinates_list:
    lon, lat = coords[0], coords[1]
    buffer_size = (pixel_size * export_scale) / 2
    point = Geometry.Point([lon, lat])
    bbox = point.buffer(buffer_size).bounds()

    hlss_image_collection: ImageCollection = ImageCollection(image_collection_hlss).filterBounds(bbox).filterDate(start_date, end_date).select(bands_to_export_hlss)
    hlss_image_collection = hlss_image_collection.map(lambda image: image.rename(list(map_hlss.keys())))

    hlsl_image_collection: ImageCollection = ImageCollection(image_collection_hlsl).filterBounds(bbox).filterDate(start_date, end_date).select(bands_to_export_hlsl)
    hlsl_image_collection = hlsl_image_collection.map(lambda image: image.rename(list(map_hlsl.keys())))

    hls_image_collection: ImageCollection = hlss_image_collection.merge(hlsl_image_collection).sort("system:time_start")

    collection_size = hls_image_collection.size().getInfo()
    if collection_size == 0:
        print(f"Skipping empty collection for coords {lon}, {lat}")
        continue

    hls_image_collection_remove_clouds = hls_image_collection.map(mask_hls_clouds)

    # Convert to a list to slice into chunks of max_bands_per_export
    collection_list = hls_image_collection_remove_clouds.toList(collection_size)

    for chunk_idx, start_idx in enumerate(range(0, collection_size, max_bands_per_export)):
        end_idx = min(start_idx + max_bands_per_export, collection_size)
        chunk_list = collection_list.slice(start_idx, end_idx)
        chunk_collection = ImageCollection(chunk_list)

        for band_name in export_bands:
            time_series_cube = chunk_collection.select(band_name).toBands()

            collection_name_for_file = "NASA_HLS_v002"
            # Appended chunk index part to the file name to avoid overwrite
            file_name = f"{collection_name_for_file}_{band_name}_lon{lon:.4f}_lat{lat:.4f}_part{chunk_idx+1}"
            if retry_list and file_name not in retry_list:
                continue
            task = Export.image.toDrive(
                image=time_series_cube.toFloat(),
                description=file_name,
                folder=export_folder,
                fileNamePrefix=file_name,
                region=bbox,
                scale=export_scale,
                crs="EPSG:4326",
                maxPixels=1e10,
            )
            task.start()
            print(f"Started export task: {file_name}")
